## Total number of Users

In [19]:
import pandas as pd
from pathlib import Path

# --------------------------
# West Bay vs East Bay station sets
# --------------------------
WEST_BAY = {
    # Abbreviations
    "EMBR","MONT","POWL","CIVC","16TH","24TH","GLEN","BALB","DALY","COLM","SSAN","SBRN","MLBR","SFIA",
    # Full names
    "Embarcadero","Montgomery St.","Powell St.","Civic Center/UN Plaza","16th St. Mission","24th St. Mission",
    "Glen Park","Balboa Park","Daly City","Colma","South San Francisco","San Bruno","Millbrae",
    "San Francisco International Airport"
}
EAST_BAY = {
    # Abbreviations
    "WOAK","12TH","19TH","MCAR","LAKE","FTVL","COLS","SANL","BAYF","HAYW","SHAY","UCTY","FRMT","WARM",
    "MLPT","BERY","ROCK","ORIN","LAFY","WCRK","PHIL","CONC","NCON","PITT","PCTR","ANTC",
    "RICH","DELN","PLZA","NBRK","DBRK","ASHB","OAKL",
    # Full names
    "West Oakland","12th St. Oakland City Center","19th St. Oakland","MacArthur","Lake Merritt","Fruitvale",
    "Coliseum","San Leandro","Bay Fair","Hayward","South Hayward","Union City","Fremont","Warm Springs/South Fremont",
    "Milpitas","Berryessa/North San José","Rockridge","Orinda","Lafayette","Walnut Creek",
    "Pleasant Hill/Contra Costa Centre","Concord","North Concord/Martinez","Pittsburg/Bay Point",
    "Pittsburg Center","Antioch","Richmond","El Cerrito del Norte","El Cerrito Plaza",
    "North Berkeley","Downtown Berkeley","Ashby","Oakland International Airport"
}

# ---- helpers ----
def _norm_station(s):
    if pd.isna(s):
        return None
    return str(s).strip()

def _in_set(station, sset):
    if station is None:
        return False
    if station in sset:
        return True
    alt = station.replace(".", "")
    if alt in sset:
        return True
    alt2 = alt.replace(" St ", " St. ")
    if alt2 in sset:
        return True
    alt3 = alt2.replace(" / ", "/").replace("  ", " ")
    return alt3 in sset

def _read_one_csv(path, assume_no_header=True, column_names=None, use_first_n_cols=5):
    if assume_no_header:
        if column_names is None:
            column_names = ["date","hour","origin","destination","ridership"]
        return pd.read_csv(
            path,
            header=None,
            names=column_names,
            usecols=list(range(use_first_n_cols))
        )
    else:
        return pd.read_csv(path)

def _load_csvs(path_or_dir, assume_no_header=True, column_names=None, use_first_n_cols=5):
    p = Path(path_or_dir)
    if p.is_dir():
        files = sorted(p.glob("*.csv"))
        if not files:
            raise FileNotFoundError(f"No CSV files found in directory: {p}")
        return pd.concat(
            [_read_one_csv(f, assume_no_header, column_names, use_first_n_cols) for f in files],
            ignore_index=True
        )
    return _read_one_csv(p, assume_no_header, column_names, use_first_n_cols)

def average_weekday_transbay_morning_aug2025(
    path_or_dir,
    assume_no_header=True,
    column_names=None,
    use_first_n_cols=5,
    include_us_holidays=False,  # if True, drop US federal holidays that fall on weekdays
    use_calendar_weekdays=False  # if True, average over *all* weekdays in Aug 2025 (fill missing dates with 0)
):
    """
    Returns the average *weekday* (Mon–Fri) ridership crossing between East Bay and West Bay
    during 05:00–09:59 in August 2025.

    Parameters
    ----------
    path_or_dir : str or Path
        CSV file or directory of CSVs.
    assume_no_header : bool
        Treat files as headerless and assign `column_names`.
    column_names : list[str]
        Defaults to ["date","hour","origin","destination","ridership"].
    use_first_n_cols : int
        Number of leading columns to read when headerless.
    include_us_holidays : bool
        If True, exclude US federal holidays from weekdays (requires 'holidays' pkg if you later add it).
        Currently only Mon–Fri filtering is applied; set False to keep all Mon–Fri.
    use_calendar_weekdays : bool
        If True, average over the full set of Mon–Fri dates in Aug 2025,
        filling any missing dates with 0. If False, average only over dates present in data.
    """
    if column_names is None:
        column_names = ["date","hour","origin","destination","ridership"]

    df = _load_csvs(path_or_dir, assume_no_header, column_names, use_first_n_cols).copy()

    # Parse date/hour; filter to Aug 2025 and 5:00–9:59
    df["_date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[(df["_date"].dt.year == 2025) & (df["_date"].dt.month == 8)]
    df["_hour"] = pd.to_numeric(df["hour"], errors="coerce")
    df = df[df["_hour"].between(5, 9, inclusive="both")]  # 5,6,7,8,9

    # Weekdays (Mon=0,...,Sun=6) -> keep 0..4
    df = df[df["_date"].dt.weekday <= 4]

    # (Optional) US holiday exclusion hook — left as a no-op unless you wire in a holiday calendar
    if include_us_holidays:
        # Example stub (requires `pip install holidays` to make real):
        # import holidays
        # us_holidays = holidays.US(years=[2025])
        # df = df[~df["_date"].isin([d for d in df["_date"].dt.date if d in us_holidays])]
        pass

    # Normalize stations & flag cross-bay ODs
    df["_o"] = df["origin"].apply(_norm_station)
    df["_d"] = df["destination"].apply(_norm_station)
    df["_o_west"] = df["_o"].apply(lambda s: _in_set(s, WEST_BAY))
    df["_o_east"] = df["_o"].apply(lambda s: _in_set(s, EAST_BAY))
    df["_d_west"] = df["_d"].apply(lambda s: _in_set(s, WEST_BAY))
    df["_d_east"] = df["_d"].apply(lambda s: _in_set(s, EAST_BAY))
    crosses = (df["_o_west"] & df["_d_east"]) | (df["_o_east"] & df["_d_west"])
    df = df[crosses].copy()

    # Daily sums, then mean across weekdays
    df["_riders"] = pd.to_numeric(df["ridership"], errors="coerce").fillna(0)
    daily = df.groupby(df["_date"].dt.date)["_riders"].sum().rename("daily_total")

    if use_calendar_weekdays:
        # Average over *all* calendar weekdays in Aug 2025; fill missing days with 0
        idx = pd.bdate_range(start="2025-08-01", end="2025-08-31", freq="C")  # business days (Mon–Fri)
        idx = pd.Index([d.date() for d in idx])
        daily = daily.reindex(idx, fill_value=0)

    print(df.head())

    avg_weekday = float(daily.mean()) if len(daily) else 0.0

    return {
        "average_weekday_riders": avg_weekday,
        "num_weekdays_counted": int(len(daily)),
        "dates_covered": sorted(map(str, daily.index))  # helpful for auditing
    }

In [29]:
result = average_weekday_transbay_morning_aug2025(
     "date-hour-soo-dest-2025.csv",
     assume_no_header=True,
     column_names=["date","hour","origin","destination","ridership"],
     use_first_n_cols=5,
     include_us_holidays=False,
     use_calendar_weekdays=False
 )
print(result)

               date  hour origin destination  ridership      _date  _hour  \
5325185  2025-08-01     5   12TH        16TH          1 2025-08-01      5   
5325186  2025-08-01     5   12TH        24TH          2 2025-08-01      5   
5325187  2025-08-01     5   12TH        CIVC          4 2025-08-01      5   
5325188  2025-08-01     5   12TH        EMBR          3 2025-08-01      5   
5325189  2025-08-01     5   12TH        MONT          2 2025-08-01      5   

           _o    _d  _o_west  _o_east  _d_west  _d_east  _riders  
5325185  12TH  16TH    False     True     True    False        1  
5325186  12TH  24TH    False     True     True    False        2  
5325187  12TH  CIVC    False     True     True    False        4  
5325188  12TH  EMBR    False     True     True    False        3  
5325189  12TH  MONT    False     True     True    False        2  
{'average_weekday_riders': 28519.04761904762, 'num_weekdays_counted': 21, 'dates_covered': ['2025-08-01', '2025-08-04', '2025-08-05', '

In [21]:
df = pd.read_csv('date-hour-soo-dest-2025.csv')

In [22]:
df.head()

,2025-01-01,0,12TH,16TH,1
0,2025-01-01,0,12TH,ANTC,1
1,2025-01-01,0,12TH,BALB,1
2,2025-01-01,0,12TH,BERY,1
3,2025-01-01,0,12TH,CIVC,8
4,2025-01-01,0,12TH,COLS,2


In [25]:
import pandas as pd
from pathlib import Path

def average_weekday_morning_flow_pems(
    file_path,
    year=2025,
    month=8,
    start_hour=5,
    end_hour=9,  # inclusive => 5:00–9:59
    min_observed_pct=None  # e.g., 80 to require >=80% observed per row; None to skip
):
    """
    Compute the average weekday morning (start_hour–end_hour) total flow for a PeMS export.

    Parameters
    ----------
    file_path : str or Path
        Path to the PeMS Excel/CSV. Must contain columns:
        - 'Hour' (timestamps)
        - 'Flow (Veh/Hour)' (integer/float per hour)
        - '% Observed' (optional; filter with min_observed_pct if provided)
    year, month : int
        Target month (e.g., 2025, 8 for August 2025).
    start_hour, end_hour : int
        Hours to include, inclusive (e.g., 5..9 for 5–9 AM).
    min_observed_pct : int or float or None
        If set (e.g., 80), only keep rows with '% Observed' >= this value.

    Returns
    -------
    result : dict
        {
          "average_weekday_morning_flow": float,
          "num_weekdays_counted": int,
          "daily_totals": dict[YYYY-MM-DD -> float]
        }
    """
    file_path = Path(file_path)

    # Read Excel/CSV (auto-detect by suffix)
    if file_path.suffix.lower() in {".xls", ".xlsx"}:
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path)

    # Basic sanity: required columns
    required = {"Hour", "Flow (Veh/Hour)"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Found: {list(df.columns)}")

    # Parse timestamps
    df = df.copy()
    df["Hour"] = pd.to_datetime(df["Hour"], errors="coerce")
    df = df.dropna(subset=["Hour"])

    # Filter target month & weekdays (Mon=0..Sun=6)
    df = df[(df["Hour"].dt.year == year) & (df["Hour"].dt.month == month)]
    df = df[df["Hour"].dt.weekday <= 4]

    # Optional data coverage filter
    if (min_observed_pct is not None) and ("% Observed" in df.columns):
        df = df[pd.to_numeric(df["% Observed"], errors="coerce").fillna(0) >= float(min_observed_pct)]

    # Time-of-day window (inclusive hours)
    df = df[df["Hour"].dt.hour.between(start_hour, end_hour, inclusive="both")]

    # Sum per date
    df["_date"] = df["Hour"].dt.date
    daily_totals = (
        df.groupby("_date")["Flow (Veh/Hour)"]
          .sum()
          .rename("daily_total")
          .sort_index()
    )

    avg_flow = float(daily_totals.mean()) if len(daily_totals) else 0.0

    return {
        "average_weekday_morning_flow": avg_flow,
        "num_weekdays_counted": int(len(daily_totals)),
        "daily_totals": {d.isoformat(): float(v) for d, v in daily_totals.items()},
    }

In [26]:
result = average_weekday_morning_flow_pems(
     "pems_output.xlsx",
     year=2025, month=8,
     start_hour=5, end_hour=9,
     min_observed_pct=None  # set to e.g. 80 if you want a coverage filter
 )
print(f"Average weekday 5–10AM flow: {result['average_weekday_morning_flow']:.0f} vehicles")
print(f"Weekdays counted: {result['num_weekdays_counted']}")
print("Daily totals:", result["daily_totals"])

Average weekday 5–10AM flow: 41369 vehicles
Weekdays counted: 21
Daily totals: {'2025-08-01': 40201.0, '2025-08-04': 40031.0, '2025-08-05': 41891.0, '2025-08-06': 41570.0, '2025-08-07': 40541.0, '2025-08-08': 40988.0, '2025-08-11': 41325.0, '2025-08-12': 42480.0, '2025-08-13': 41835.0, '2025-08-14': 42369.0, '2025-08-15': 40867.0, '2025-08-18': 41835.0, '2025-08-19': 41895.0, '2025-08-20': 39643.0, '2025-08-21': 41264.0, '2025-08-22': 40866.0, '2025-08-25': 42154.0, '2025-08-26': 42737.0, '2025-08-27': 41858.0, '2025-08-28': 42976.0, '2025-08-29': 39426.0}


In [41]:
27132 + 41369

68501

In [43]:
68501 / 5

13700.2

## Average Fare and Travel Time in BART

In [34]:
import pandas as pd

# ---------------------------------------------------
# 1. Paths
# ---------------------------------------------------
fare_rules_path       = "fare_rules.txt"
fare_attributes_path  = "fare_attributes.txt"
od_ridership_path     = "date-hour-soo-dest-2025.csv"

# ---------------------------------------------------
# 2. Load datasets
# ---------------------------------------------------
fare_rules = pd.read_csv(fare_rules_path)
fare_attr  = pd.read_csv(fare_attributes_path)

od = pd.read_csv(
    od_ridership_path,
    header=None,
    names=["date","hour","origin","destination","riders"]
)
od["date"] = pd.to_datetime(od["date"], errors="coerce")
od["hour"] = pd.to_numeric(od["hour"], errors="coerce")
od["riders"] = pd.to_numeric(od["riders"], errors="coerce").fillna(0)

# ---------------------------------------------------
# 3. Filter to August 2025, weekdays (Mon–Fri), 5–10 AM
# ---------------------------------------------------
od = od[(od["date"].dt.year == 2025) & (od["date"].dt.month == 8)]
od["weekday"] = od["date"].dt.weekday          # Monday=0 ... Sunday=6
od = od[od["weekday"] <= 4]                    # keep Mon–Fri only
od = od[od["hour"].between(5, 9, inclusive="both")]
od = od[od["riders"] > 0]

# ---------------------------------------------------
# 4. Define East-Bay / West-Bay station sets
# ---------------------------------------------------
west_bay = {
    "EMBR","MONT","POWL","CIVC","16TH","24TH","GLEN","BALB",
    "DALY","COLM","SSAN","SBRN","SFIA","MLBR"
}
east_bay = {
    "WOAK","12TH","19TH","MCAR","LAKE","FTVL","COLS","SANL",
    "BAYF","HAYW","SHAY","UCTY","FRMT","WARM","MLPT","BERY",
    "CAST","DUBL","WDUB","RICH","DELN","PLZA","NBRK","DBRK",
    "ASHB","ORIN","LAFY","WCRK","PHIL","CONC","NCON","PITT",
    "PCTR","ANTC","OAKL", "ROCK"
}

od = od[od["origin"].isin(east_bay) & od["destination"].isin(west_bay)].copy()

# ---------------------------------------------------
# 5. Merge fares
# ---------------------------------------------------
fares = fare_rules.merge(
    fare_attr[["fare_id","price"]],
    on="fare_id", how="left"
).rename(columns={
    "origin_id":"origin",
    "destination_id":"destination",
    "price":"fare_usd"
})

merged = od.merge(
    fares[["origin","destination","fare_usd"]],
    on=["origin","destination"], how="left"
).dropna(subset=["fare_usd"])

merged["fare_usd"] = pd.to_numeric(merged["fare_usd"], errors="coerce")

# ---------------------------------------------------
# 6. Weighted-average fare and total weekday riders
# ---------------------------------------------------
total_riders = merged["riders"].sum()

daily_riders = merged.groupby(merged["date"].dt.date)["riders"].sum()
avg_weekday_riders = daily_riders.mean()

avg_fare = (merged["fare_usd"] * merged["riders"]).sum() / total_riders

print(f"Weighted-average Transbay fare (Weekdays Aug 2025 5–10 AM): ${avg_fare:,.2f}")
print(f"Average weekday East→West riders (Aug 2025 5–10 AM): {int(avg_weekday_riders):,}")

# ---------------------------------------------------
# 7. (Optional) show top 10 contributing ODs
# ---------------------------------------------------
top10 = (merged.groupby(["origin","destination"])["riders"]
              .sum()
              .sort_values(ascending=False)
              .head(10))
print("\nTop 10 East→West weekday ODs (Aug 2025 5–10 AM):")
print(top10)


Weighted-average Transbay fare (Weekdays Aug 2025 5–10 AM): $6.14
Average weekday East→West riders (Aug 2025 5–10 AM): 27,132

Top 10 East→West weekday ODs (Aug 2025 5–10 AM):
origin  destination
WOAK    EMBR           13736
DUBL    EMBR           11493
WCRK    EMBR           10043
PHIL    EMBR            9648
ROCK    EMBR            8887
WOAK    MONT            8875
DUBL    MONT            8540
PHIL    MONT            8098
19TH    EMBR            7976
LAFY    EMBR            7750
Name: riders, dtype: int64


In [44]:
# Fix: map station ABBREVIATIONS in your OD file to FULL station names/IDs in GTFS,
# then compute rider-weighted average scheduled IVT (min) for East→West trips
# on Yellow-S, Green-S, Red-S, Blue-S during Aug-2025 weekdays 5–10AM.

import pandas as pd
import numpy as np
from pathlib import Path

# -----------------------------
# Paths
# -----------------------------
TRIPS_PATH       = Path("trips.txt")
STOP_TIMES_PATH  = Path("stop_times.txt")
ROUTES_PATH      = Path("routes.txt")
STOPS_PATH       = Path("stops.txt")          # <— REQUIRED for name mapping
OD_PATH          = Path("date-hour-soo-dest-2025.csv")

# -----------------------------
# Load GTFS + OD
# -----------------------------
trips = pd.read_csv(TRIPS_PATH)
stop_times = pd.read_csv(STOP_TIMES_PATH)
routes = pd.read_csv(ROUTES_PATH)
stops = pd.read_csv(STOPS_PATH)  # must contain at least: stop_id, stop_name (and/or stop_code)

od = pd.read_csv(OD_PATH, header=None,
                 names=["date","hour","origin","destination","riders"])
od["date"]   = pd.to_datetime(od["date"], errors="coerce")
od["hour"]   = pd.to_numeric(od["hour"], errors="coerce")
od["riders"] = pd.to_numeric(od["riders"], errors="coerce").fillna(0)

# -----------------------------
# Temporal filter: Aug 2025, weekdays, 5–10AM
# -----------------------------
od = od[(od["date"].dt.year==2025) & (od["date"].dt.month==8)]
od = od[od["date"].dt.weekday <= 4]
od = od[od["hour"].between(5,9)]
od = od[od["riders"] > 0]

# -----------------------------
# Canonical East / West sets (ABBREVIATIONS as used in OD)
# -----------------------------
WEST_BAY = {
    "EMBR","MONT","POWL","CIVC","16TH","24TH","GLEN","BALB",
    "DALY","COLM","SSAN","SBRN","SFIA","MLBR"
}
EAST_BAY = {
    "WOAK","12TH","19TH","MCAR","LAKE","FTVL","COLS","SANL",
    "BAYF","HAYW","SHAY","UCTY","FRMT","WARM","MLPT","BERY",
    "CAST","DUBL","WDUB","RICH","DELN","PLZA","NBRK","DBRK",
    "ASHB","ROCK","ORIN","LAFY","WCRK","PHIL","CONC","NCON",
    "PITT","PCTR","ANTC","OAKL"
}
od = od[od["origin"].isin(EAST_BAY) & od["destination"].isin(WEST_BAY)].copy()

# -----------------------------
# Abbrev → Full-name mapping for BART (covers all above codes)
# You can extend if needed; these are the standard BART names found in stops.txt
# -----------------------------
ABBR_TO_FULL = {
    "EMBR":"Embarcadero", "MONT":"Montgomery St", "POWL":"Powell St", "CIVC":"Civic Center",
    "16TH":"16th St Mission", "24TH":"24th St Mission", "GLEN":"Glen Park", "BALB":"Balboa Park",
    "DALY":"Daly City", "COLM":"Colma", "SSAN":"South San Francisco", "SBRN":"San Bruno",
    "SFIA":"San Francisco International Airport", "MLBR":"Millbrae",

    "WOAK":"West Oakland", "12TH":"12th St Oakland City Center", "19TH":"19th St Oakland",
    "MCAR":"MacArthur", "LAKE":"Lake Merritt", "FTVL":"Fruitvale", "COLS":"Coliseum",
    "SANL":"San Leandro", "BAYF":"Bay Fair", "HAYW":"Hayward", "SHAY":"South Hayward",
    "UCTY":"Union City", "FRMT":"Fremont", "WARM":"Warm Springs/South Fremont",
    "MLPT":"Milpitas", "BERY":"Berryessa/North San Jose",
    "CAST":"Castro Valley", "DUBL":"Dublin/Pleasanton", "WDUB":"West Dublin/Pleasanton",
    "RICH":"Richmond", "DELN":"El Cerrito del Norte", "PLZA":"El Cerrito Plaza",
    "NBRK":"North Berkeley", "DBRK":"Downtown Berkeley", "ASHB":"Ashby",
    "ROCK":"Rockridge", "ORIN":"Orinda", "LAFY":"Lafayette", "WCRK":"Walnut Creek",
    "PHIL":"Pleasant Hill/Contra Costa Centre", "CONC":"Concord", "NCON":"North Concord/Martinez",
    "PITT":"Pittsburg/Bay Point", "PCTR":"Pittsburg Center", "ANTC":"Antioch",
    "OAKL":"Oakland International Airport"
}

# Create helper to map an ABBREV to all stop_ids belonging to that station (any platform variant)
# We match by stop_name containing the full station name (case-insensitive).
def stop_ids_for_abbrev(abbr):
    full = ABBR_TO_FULL.get(abbr)
    if full is None:
        return set()
    mask = stops["stop_name"].str.contains(full, case=False, na=False)
    return set(stops.loc[mask, "stop_id"].astype(str))

# Precompute stop_id sets for all abbrevs that appear in our filtered OD
all_abbrs = set(od["origin"]).union(set(od["destination"]))
ABBR_TO_STOPIDS = {abbr: stop_ids_for_abbrev(abbr) for abbr in all_abbrs}

# -----------------------------
# Route short names for cross-bay southbound lines
# -----------------------------
# ensure both keys are the same dtype (strings), then merge
trips["route_id"]  = trips["route_id"].astype(str)
routes["route_id"] = routes["route_id"].astype(str)

# some GTFS exports use mixed-case short names; standardize for matching (e.g., "Yellow-S")
routes["route_short_name"] = routes["route_short_name"].astype(str).str.strip()

# now merge safely
routes = routes.rename(columns={"route_short_name": "route_sn"})
trips  = trips.merge(routes[["route_id", "route_sn"]], on="route_id", how="left")
CROSSBAY = {"Yellow-S","Green-S","Red-S","Blue-S"}

# -----------------------------
# Time utilities
# -----------------------------
def hhmmss_to_sec(t):
    if pd.isna(t): return np.nan
    h,m,s = map(int, str(t).split(":"))
    return h*3600 + m*60 + s

stop_times["arr_sec"] = stop_times["arrival_time"].apply(hhmmss_to_sec)
stop_times["dep_sec"] = stop_times["departure_time"].apply(hhmmss_to_sec)

# Keep only trips on the four cross-bay southbound lines
stop_times = stop_times.merge(trips[["trip_id","route_sn"]], on="trip_id", how="left")
stop_times = stop_times[stop_times["route_sn"].isin(CROSSBAY)].copy()

# -----------------------------
# Compute scheduled IVT for each OD pair:
# For each trip on cross-bay lines, if it contains an origin stop_id (any platform/id variant)
# AND a destination stop_id with stop_sequence later than origin, take the time difference.
# Then average across all such trips whose *origin departure time* is between 05:00 and 09:59.
# -----------------------------
def od_ivt_minutes_for_pair(o_abbr, d_abbr):
    o_ids = ABBR_TO_STOPIDS.get(o_abbr, set())
    d_ids = ABBR_TO_STOPIDS.get(d_abbr, set())
    if not o_ids or not d_ids:
        return np.nan

    st = stop_times.copy()
    # mark origin/dest rows
    st["is_o"] = st["stop_id"].astype(str).isin(o_ids)
    st["is_d"] = st["stop_id"].astype(str).isin(d_ids)

    # origin rows per trip
    o_rows = st[st["is_o"]][["trip_id","stop_sequence","dep_sec"]].rename(
        columns={"stop_sequence":"o_seq","dep_sec":"o_dep"})
    d_rows = st[st["is_d"]][["trip_id","stop_sequence","arr_sec"]].rename(
        columns={"stop_sequence":"d_seq","arr_sec":"d_arr"})

    paired = o_rows.merge(d_rows, on="trip_id", how="inner")
    # require destination after origin
    paired = paired[paired["d_seq"] > paired["o_seq"]]

    # limit origin departure to 05:00–09:59
    paired = paired[(paired["o_dep"] >= 5*3600) & (paired["o_dep"] < 10*3600)]
    if paired.empty:
        return np.nan

    ivt_min = (paired["d_arr"] - paired["o_dep"]) / 60.0
    # drop negative/na
    ivt_min = ivt_min.replace([np.inf, -np.inf], np.nan).dropna()
    ivt_min = ivt_min[ivt_min >= 0]
    if ivt_min.empty:
        return np.nan
    return float(ivt_min.mean())

# Build an IVT (min) column for each OD row (vectorized over unique pairs to keep it fast)
unique_pairs = od[["origin","destination"]].drop_duplicates()
unique_pairs["ivt_min"] = unique_pairs.apply(
    lambda r: od_ivt_minutes_for_pair(r["origin"], r["destination"]), axis=1
)

od = od.merge(unique_pairs, on=["origin","destination"], how="left")
od = od.dropna(subset=["ivt_min"])
od = od[od["ivt_min"] > 0]

# -----------------------------
# Rider-weighted average by route and overall
# We can infer route by the origin station’s line; define a robust map (ABBREVS).
# -----------------------------
ROUTE_MAP = {
    "Yellow-S": {"ANTC","PCTR","PITT","NCON","CONC","PHIL","WCRK","LAFY","ORIN","ROCK","MCAR","19TH","12TH","WOAK"},
    "Green-S":  {"BERY","MLPT","WARM","FRMT","UCTY","SHAY","HAYW","BAYF","SANL","COLS","FTVL","LAKE","12TH","WOAK"},
    "Red-S":    {"RICH","DELN","PLZA","NBRK","DBRK","ASHB","MCAR","19TH","12TH","WOAK"},
    "Blue-S":   {"DUBL","WDUB","CAST","BAYF","SANL","COLS","FTVL","LAKE","12TH","WOAK"},
}
def assign_route_from_origin(abbr):
    for r, stns in ROUTE_MAP.items():
        if abbr in stns: return r
    return np.nan

od["route_sn"] = od["origin"].apply(assign_route_from_origin)
od = od[od["route_sn"].isin(CROSSBAY)]

# weighted averages
route_avg = (
    od.groupby("route_sn")
      .apply(lambda df: np.average(df["ivt_min"], weights=df["riders"]))
      .reset_index(name="weighted_avg_ivt_min")
      .sort_values("route_sn")
)
overall = float(np.average(od["ivt_min"], weights=od["riders"]))

print("Rider-weighted scheduled IVT (min), East→West, Aug-2025 weekdays 5–10AM:")
print(route_avg.to_string(index=False))
print(f"\nOverall weighted average: {overall:.1f} minutes")

# (Optional) sanity checks
missing_o = [a for a,sids in ABBR_TO_STOPIDS.items() if not sids]
if missing_o:
    print("\nNote: no GTFS stop_ids matched for these station codes:", missing_o)


Rider-weighted scheduled IVT (min), East→West, Aug-2025 weekdays 5–10AM:
route_sn  weighted_avg_ivt_min
  Blue-S             34.453590
 Green-S             30.457021
   Red-S             33.007745
Yellow-S             31.982404

Overall weighted average: 31.8 minutes

Note: no GTFS stop_ids matched for these station codes: ['DUBL', 'NCON', 'WDUB', 'WARM', 'PHIL', '16TH', 'BERY', '24TH', '12TH', 'PITT', '19TH']


/tmp/ipython-input-1177220711.py:194: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: np.average(df["ivt_min"], weights=df["riders"]))
